In [1]:
from graph import app 
import pandas as pd
import os
from tqdm.auto import tqdm
import torch
import time 
from Configuration.models import models #import models to evaluate

# Evaluate models

For Gemma 9b add : 


if database_name == 'offshoreleaks':

print(f'Skipping row {i}, to avoid memory error in {database_name}')

continue

In [ ]:
def evaluate_model (model_name, model, dataset):
    ''' 
    Evaluate the selected model on the given dataset, by invoking the graph workflow.
    Save the dataframe with the questions and the generated cypher queries to the Outputs_Evaluation folder.
    Return the dataframe
    '''
    # torch.cuda.empty_cache()#clear gpu cache
    # Get datasets 
    if dataset =='sustain':
        df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx',sheet_name='Queries')
        subfolder = 'SustainDataset'
    elif dataset == 'benchmark2025':
        df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2025'
    elif dataset == 'benchmark2024':
        df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2024'
    
    output_dir = f"Outputs_Evaluation/{subfolder}/{model_name}/"
    # Create the directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    results_path_pkl = os.path.join(output_dir, f"{model_name}.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{model_name}.xlsx")
   
    # Check if results file already exists
    if os.path.exists(results_path_xlsx):
        df_results = pd.read_excel(results_path_xlsx)
    else:# Run model on dataset
        df_results = df.copy() # Copy of the df with questions and expected answers, to add the answers of the llm 
        # Now define additional columns in df_results to store LLM results:
        df_results['generated_cypher'] = None 
        df_results['answer'] = None
        df_results['path'] = None           # Use None so we can store a list
        df_results['output'] = None           # Use None so we can store a list
        df_results['cypher_errors_history'] = None           # Use None so we can store a list
        df_results['cypher_errors'] = None           # Use None so we can store a list
        df_results['is_KG_Valid_correct'] = None           # Use None so we can store a list
        df_results['total_tokens'] = None           # Use None so we can store a list
        df_results['completion_tokens'] = None           # Use None so we can store a list
        df_results['prompt_tokens'] = None           # Use None so we can store a list

    config = {'configurable': {'model': model, 'model_name':model_name, 'attempts_w_filtered':5, 'total_attempts':10},'recursion_limit':50}
    
    print(f'Starting evaluation for {model_name} on {dataset} ...')
    # Iterate over each row and compute only if not already done
    for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
        if pd.isna(row['generated_cypher']):
            question = row['Question'] if dataset == 'sustain' else row['question']
            print(i,question)
            
            # Get credentials
            required_cols = {'database_name', 'url', 'username', 'password'}
            if required_cols.issubset(df_results.columns):
                database_url = row['url']
                database_name = row['database_name']
                database_user = row['username']
                database_password = row['password']
            else:
                #its the sustain dataset
                database_url = os.getenv('database_url')
                database_name = os.getenv('database_name')
                database_user = os.getenv('database_user')
                database_password = os.getenv('database_password')
                # activate it for gemma 9b
                # if database_name == 'offshoreleaks':
                #     print(f'Skipping row {i}, to avoid memory error in {database_name}')
                #     continue
                # else:# Configuration 
                inputs = {
                    'question':question,
                    'database_url': database_url,
                    'database_name':database_name,
                    'database_user':database_user,
                    'database_password':database_password
                }
                # torch.cuda.empty_cache() #clear gpu cache
            
                # Run the graph and save outputs
                result =app.invoke(inputs, config =config) #,debug=False
                df_results.at[i, 'generated_cypher'] = result['cypher_statement']
                df_results.at[i, 'output'] = str(result['database_records'])
                df_results.at[i, 'path'] = result['path']
                df_results.at[i, 'cypher_errors'] = result['cypher_errors']
                df_results.at[i, 'cypher_errors_history'] = result['cypher_errors_history']
                df_results.at[i, 'is_KG_Valid_correct'] = result['is_KG_Valid_correct']
                df_results.at[i, 'answer'] = result['answer']
                df_results.at[i, 'total_tokens'] = result['total_tokens']
                df_results.at[i, 'completion_tokens'] = result['completion_tokens']
                df_results.at[i, 'prompt_tokens'] = result['prompt_tokens']
                if ('schema_retriever' in result['path']) and result['is_KG_Valid_correct']:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = True
                else:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = False
                torch.cuda.empty_cache()#clear gpu cache
                
                # Save results
                df_results.to_pickle(results_path_pkl)
                df_results.to_excel(results_path_xlsx, index=False)# Set index=False to exclude the index column in Excel
        else:
            print(f"Skipping row {i}, already computed.")
            
    return df_results

In [ ]:
model_name = 'llama3_2_3b'
df_results = evaluate_model (model_name, models[model_name](), 'benchmark2025')

Starting evaluation for llama3_2_3b on benchmark2025 ...


  0%|          | 0/2128 [00:00<?, ?it/s]

Skipping row 0, already computed.
Skipping row 1, already computed.
Skipping row 2, already computed.
Skipping row 3, already computed.
Skipping row 4, already computed.
Skipping row 5, already computed.
Skipping row 6, already computed.
Skipping row 7, already computed.
Skipping row 8, already computed.
Skipping row 9, already computed.
Skipping row 10, already computed.
Skipping row 11, already computed.
Skipping row 12, already computed.
Skipping row 13, already computed.
Skipping row 14, already computed.
Skipping row 15, already computed.
Skipping row 16, already computed.
Skipping row 17, already computed.
Skipping row 18, already computed.
Skipping row 19, already computed.
Skipping row 20, already computed.
Skipping row 21, already computed.
Skipping row 22, already computed.
Skipping row 23, already computed.
Skipping row 24, already computed.
Skipping row 25, already computed.
Skipping row 26, already computed.
Skipping row 27, already computed.
Skipping row 28, already compu

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: stripped_off_date)} {position: line: 1, column: 43, offset: 42} for query: 'RETURN exists { MATCH (x:Entity) WHERE (x.stripped_off_date) IS NOT NULL} AS exists'


CyVer Errors from generated query:
- Neo.ClientError.Statement.SyntaxError: Invalid input '}': expected 'WHERE' or ']' (line 1, column 93 (offset: 92))


- UnknownPathWarning: The specified path (:Address)-[:connected_to]->() does not exist in the knowledge graph schema. The path (:Address)-[:connected_to {valid_until: 'YYYY-MM-DD'}]->() in the Cypher statement is wrong.

- Neo.ClientNotification.Statement.UnknownPropertyKeyWarning: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: stripped_off_date)

- InvalidPropertyAccessByLabelWarning: The label Entity does not have the following properties:  country, stripped_off_date.

Cypher Corrector: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (e:Entity {countries: 'Samoa'})
WHERE EXISTS ((e)-[:registered_address {valid_until: 'YYYY-MM-DD'}]->(:Address))
RETURN avg(CAS

...

CyVer Errors from generated query:
- WrongDirectionPathWarning: The specified path (:Genre)-[:IN_GENRE]->(:Movie) has the opposite direction in the knowledge graph schema. The path (g)-[:IN_GENRE {genre: 'Action'}]->(m) in the Cypher statement should be changed to  (g)<-[:IN_GENRE {genre: 'Action'}]-(m).

- Neo.ClientNotification.Statement.UnknownPropertyKeyWarning: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: genre)

- InvalidPropertyAccessByLabelWarning: The label IN_GENRE does not have the following properties:  genre.

Cypher Corrector: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (g:Genre {name: 'Action'})
CREATE (m:Movie {url: 'https://example.com/movie', poster: 'poster.png', runtime: 120, budget: 10000000, revenue: 50000000})
CREATE (m)<-[:IN_GENRE {genre: 'Action'}]-(g):


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: genre)} {position: line: 1, column: 51, offset: 50} for query: 'RETURN exists { MATCH ()-[x:IN_GENRE]-() WHERE (x.genre) IS NOT NULL} AS exists'


CyVer Errors from generated query:
- WrongDirectionPathWarning: The specified path (:Movie)<-[:IN_GENRE]-(:Genre) has the opposite direction in the knowledge graph schema. The path (m)<-[:IN_GENRE {genre: 'Action'}]-(g) in the Cypher statement should be changed to  (m)-[:IN_GENRE {genre: 'Action'}]->(g).

- Neo.ClientNotification.Statement.UnknownPropertyKeyWarning: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: genre)

- InvalidPropertyAccessByLabelWarning: The label IN_GENRE does not have the following properties:  genre.

Cypher Corrector: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (g:Genre {name: 'Action'})
MERGE (m:Movie {url: 'https://example.com/movie', poster: 'poster.png', runtime: 120, budget: 10000000, revenue: 50000000})
MERGE (m)-[:IN_GENRE {genre: 'Action'}]->(g):


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: genre)} {position: line: 1, column: 51, offset: 50} for query: 'RETURN exists { MATCH ()-[x:IN_GENRE]-() WHERE (x.genre) IS NOT NULL} AS exists'


CyVer Errors from generated query:
- Neo.ClientNotification.Statement.UnknownPropertyKeyWarning: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: genre)

- InvalidPropertyAccessByLabelWarning: The label IN_GENRE does not have the following properties:  genre.

Cypher Corrector: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (g:Genre {name: 'Action'})
MERGE (m:Movie {url: 'https://example.com/movie', poster: 'poster.png', runtime: 120, budget: 10000000, revenue: 50000000})
MERGE (m)-[:IN_GENRE]->(g):
Cypher Executor: START
Output Generator: START
2125 How to remove all Movies older than the year 2000? Give me the recommended way of deleting a large volume of data
Schema filtering Agent: START
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of MATCH (m:Movie) WHERE m.year < 2000 DELETE m:
Cypher Exe

# Evaluate with computational metrics

In [2]:
def evaluate_model_usage (model_name, model, dataset):
    ''' 
    Evaluate the selected model on the given dataset, by invoking the graph workflow.
    Save the dataframe with the questions and the generated cypher queries to the Outputs_Evaluation folder.
    Return the dataframe
    '''
    # torch.cuda.empty_cache()#clear gpu cache
    # Get datasets 
    if dataset =='sustain':
        df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx',sheet_name='Queries')
        subfolder = 'SustainDataset'
    elif dataset == 'benchmark2025':
        df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2025'
    elif dataset == 'benchmark2024':
        df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2024'
    
    output_dir = f"Outputs_Evaluation/{subfolder}/{model_name}/"
    # Create the directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    results_path_pkl = os.path.join(output_dir, f"{model_name}_usage1.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{model_name}_usage1.xlsx")
   
    # Check if results file already exists
    if os.path.exists(results_path_xlsx):
        df_results = pd.read_excel(results_path_xlsx)
    else:# Run model on dataset
        df_results = df.copy() # Copy of the df with questions and expected answers, to add the answers of the llm 
        # Now define additional columns in df_results to store LLM results:
        df_results['generated_cypher'] = None 
        df_results['answer'] = None
        df_results['path'] = None           # Use None so we can store a list
        df_results['output'] = None           # Use None so we can store a list
        df_results['cypher_errors_history'] = None           # Use None so we can store a list
        df_results['cypher_errors'] = None           # Use None so we can store a list
        df_results['is_KG_Valid_correct'] = None           # Use None so we can store a list
        # timing columns
        df_results['eval_time_s'] = None              # <-- total wall-clock time per query
        # tokens columns
        df_results['total_tokens'] = None           # Use None so we can store a list
        df_results['completion_tokens'] = None           # Use None so we can store a list
        df_results['prompt_tokens'] = None           # Use None so we can store a list
    config = {'configurable': {'model': model, 'model_name':model_name, 'attempts_w_filtered':5, 'total_attempts':10},'recursion_limit':50}
    
    print(f'Starting evaluation for {model_name} on {dataset} ...')
    # Iterate over each row and compute only if not already done
    for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
        if pd.isna(row['generated_cypher']):
            question = row['Question'] if dataset == 'sustain' else row['question']
            print(i,question)
            
            # Get credentials
            required_cols = {'database_name', 'url', 'username', 'password'}
            if required_cols.issubset(df_results.columns):
                database_url = row['url']
                database_name = row['database_name']
                database_user = row['username']
                database_password = row['password']
            else:
                #its the sustain dataset
                database_url = os.getenv('database_url')
                database_name = os.getenv('database_name')
                database_user = os.getenv('database_user')
                database_password = os.getenv('database_password')
                # activate it for gemma 9b
                # if database_name == 'offshoreleaks':
                #     print(f'Skipping row {i}, to avoid memory error in {database_name}')
                #     continue
                # else:# Configuration 
                inputs = {
                    'question':question,
                    'database_url': database_url,
                    'database_name':database_name,
                    'database_user':database_user,
                    'database_password':database_password
                }
                # torch.cuda.empty_cache() #clear gpu cache
            
                # Run the graph and save outputs
                start = time.perf_counter() # start wall-clock timer
                result =app.invoke(inputs, config =config) #,debug=False
                elapsed = time.perf_counter() - start # elapsed wall-clock time
                df_results.at[i, 'eval_time_s'] = round(elapsed, 4)
                df_results.at[i, 'generated_cypher'] = result['cypher_statement']
                df_results.at[i, 'output'] = str(result['database_records'])
                df_results.at[i, 'path'] = result['path']
                df_results.at[i, 'cypher_errors'] = result['cypher_errors']
                df_results.at[i, 'cypher_errors_history'] = result['cypher_errors_history']
                df_results.at[i, 'is_KG_Valid_correct'] = result['is_KG_Valid_correct']
                df_results.at[i, 'answer'] = result['answer']
                df_results.at[i, 'total_tokens'] = result['total_tokens']
                df_results.at[i, 'completion_tokens'] = result['completion_tokens']
                df_results.at[i, 'prompt_tokens'] = result['prompt_tokens']
                if ('schema_retriever' in result['path']) and result['is_KG_Valid_correct']:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = True
                else:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = False
                torch.cuda.empty_cache()#clear gpu cache
                

                # Save results
                df_results.to_pickle(results_path_pkl)
                df_results.to_excel(results_path_xlsx, index=False)# Set index=False to exclude the index column in Excel
        else:
            print(f"Skipping row {i}, already computed.")
            
    return df_results

In [3]:
model_name = 'gpt_4o' # 'qwen2_5_Coder_32b' #'llama3_2_3b'
df_results = evaluate_model_usage (model_name, models[model_name](), 'sustain') #benchmark2025

Starting evaluation for gpt_4o on sustain ...


  0%|          | 0/52 [00:00<?, ?it/s]

0 What is the title of the goal with code 2?
Schema filtering Agent: START
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of 
MATCH (g:Goal {code: "2"}) RETURN g.title:
Cypher Executor: START
Output Generator: START
1 Which is the EUcode for the geoarea with name Greece ?
Schema filtering Agent: START
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of 
MATCH (g:GeoArea {name: "Greece"}) RETURN g.EUcode:
Cypher Executor: START
Output Generator: START
2 Which is the name of the area that has the EUcode FR?
Schema filtering Agent: START
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of 
MATCH (a:Area {EUcode: 'FR'}) RETURN a.name:
Cypher Executor: START
Output Generator: START
3 Which is the title of case study with code cs_1? 
Schema filtering Agent: START
Cypher Generator: START
CyVer Evaluator: START
CyVer Evaluation of 
MATCH (c:CaseStudy {code: 'cs_1'}) RETURN c.title:
Cypher Executor: START
Output Generator: START
4 Which are the haz

Llama 3 - GPU usage:

92.6 %

90.17 %\

74.4 % 

90

85.03

92.2

75.2

93.8

92.48

90.33

91.6

64

In [ ]:
#In the Sustain dataset, Llama 3 2 3b GPU usage:#NVIDIA L4 24GB
llama3_gpu_extended = [92.6,90.17,74.4 ,90,85.03,92.2,75.2,93.8,92.48,90.33,91.6,64]
mean_usage = sum(llama3_gpu_extended)/len(llama3_gpu_extended)
print(f'Mean GPU usage for Llama 3 2 3b extended: {mean_usage} %')
median_usage = sorted(llama3_gpu_extended)[len(llama3_gpu_extended)//2]
print(f'Median GPU usage for Llama 3 2 3b extended: {median_usage} %')

Mean GPU usage for Llama 3 2 3b extended: 85.98416666666667 %
Median GPU usage for Llama 3 2 3b extended: 90.33 %


In [ ]:
# NVIDIA L4 96GB
qwen_32_coder_gpu_extended = [98.25,
100,97.25,100,98,100,97.5,97.5,97.75,100,100,100,98,100,98,100,98,100,100,96.75,97.5,100,100,97.75,99,100,100,100,97.75,98.5,98,100,96.75,96.5]
mean_usage = sum(qwen_32_coder_gpu_extended)/len(qwen_32_coder_gpu_extended)
print(f'Mean GPU usage for Qwen coder 32b extended: {mean_usage} %')
median_usage = sorted(qwen_32_coder_gpu_extended)[len(qwen_32_coder_gpu_extended)//2]
print(f'Median GPU usage for Qwen coder 32b extended: {median_usage} %')

Mean GPU usage for Qwen coder 32b extended: 98.78676470588235 %
Median GPU usage for Qwen coder 32b extended: 99 %
